In [1]:
#Step1
import pandas as pd
import pandasql as ps
import pixiedust
import sys
sys.path.append('..')
import util
import etl
import pyarrow.parquet as pq
import pyarrow as pa
!pip install duckdb
import duckdb
import inspect

# Get the path of the imported module (etl.py)
etl_module_path = inspect.getfile(etl)
print("Path of the imported ETL module (etl.py):", etl_module_path)
util.usedatabase(spark, "real_world_data_jun_2022")

pixiedust.enableJobMonitor()

con = duckdb.connect()

Pixiedust database opened successfully
Table VERSION_TRACKER created successfully
Table METRICS_TRACKER created successfully

Share anonymous install statistics? (opt-out instructions)

PixieDust will record metadata on its environment the next time the package is installed or updated. The data is anonymized and aggregated to help plan for future releases, and records only the following values:

{
   "data_sent": currentDate,
   "runtime": "python",
   "application_version": currentPixiedustVersion,
   "space_id": nonIdentifyingUniqueId,
   "config": {
       "repository_id": "https://github.com/ibm-watson-data-lab/pixiedust",
       "target_runtimes": ["Data Science Experience"],
       "event_id": "web",
       "event_organizer": "dev-journeys"
   }
}
You can opt out by calling pixiedust.optOut() in a new cell.


Pixiedust runtime updated. Please restart kernel
Table SPARK_PACKAGES created successfully
Table USER_PREFERENCES created successfully
Table service_connections created successfully
Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
     |████████████████████████████████| 20.2 MB 7.1 MB/s eta 0:00:01
Path of the imported ETL module (etl.py): /home/o_suchsi/work/Oklahoma State/Priya/epilepsy/etl.py
Using real_world_data_jun_2022 ....
Successfully enabled Spark Job Progress Monitor


In [1]:
#Step2
Epilepsy_Combined = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/priya-smallset-finalF1_Numbered_NoNullStr.parquet")

In [2]:
#Step3
balanced_df = Epilepsy_Combined.drop("personid")

In [3]:
#Step4
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql import SparkSession
import random
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, VectorSizeHint

# Step 0: Mimicking randomSplit with seed and randomness
def probabilistic_split(df: DataFrame, fractions: list, seed=None) -> list:
    if seed is not None:
        random.seed(seed)

    cumulative_fractions = [sum(fractions[:i + 1]) for i in range(len(fractions))]
    random_col = F.rand(seed)
    df_with_random = df.withColumn("random", random_col)
    splits = []
    prev_fraction = 0
    for fraction in cumulative_fractions:
        split_df = df_with_random.filter((F.col("random") >= prev_fraction) & (F.col("random") < fraction))
        splits.append(split_df.drop("random"))
        prev_fraction = fraction
    return splits

# Train, validation, and test sampling fractions
train_fraction = 0.7
valid_fraction = 0.2
test_fraction = 0.1
fractions = [train_fraction, valid_fraction, test_fraction]
seed_value = 23
train_data, valid_data, test_data = probabilistic_split(balanced_df, fractions, seed=seed_value)

# Show class distribution in train, validation, and test datasets
train_data.groupBy('label').count().show()
valid_data.groupBy('label').count().show()
test_data.groupBy('label').count().show()

# Print counts
print("Sampled data count:", balanced_df.count())
print("Train data count:", train_data.count())
print("Validation data count:", valid_data.count())
print("Test data count:", test_data.count())

+-----+-----+
|label|count|
+-----+-----+
|  0.0|70345|
|  1.0|10738|
+-----+-----+

+-----+-----+
|label|count|
+-----+-----+
|  0.0|20222|
|  1.0| 2975|
+-----+-----+

+-----+-----+
|label|count|
+-----+-----+
|  0.0|10227|
|  1.0| 1565|
+-----+-----+

Sampled data count: 116072
Train data count: 81083
Validation data count: 23197
Test data count: 11792


In [4]:
#Step5
columns = train_data.columns
feature_cols = columns
feature_cols.remove('label')
target_col = ['label']

In [5]:
#Step6
###########################Included Standard Scalar #############################################################
from pyspark.sql import DataFrame
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, VectorSizeHint, StandardScaler

# Step 1: Define vector columns and non-vector columns based on schema
vector_cols = [col_name for col_name, dtype in train_data.dtypes if 'vector' in dtype]
non_vector_cols = [col_name for col_name in feature_cols if col_name not in vector_cols]

# Step 2: Function to apply VectorSizeHint and return size hint stages for the pipeline
def get_vector_size_hint_stage(data, col_name):
    # Sample a small portion of the data to determine vector size
    sample_fraction = 0.0001  # Using 0.01% of the data for sampling
    sampled_df = data.select(col_name).sample(False, sample_fraction).limit(1)
    sample_row = sampled_df.take(1)

    if sample_row:
        vector_size = len(sample_row[0][col_name])
        print(f"Column '{col_name}' vector size: {vector_size}")
        # Return a VectorSizeHint stage for the pipeline if vector size is valid
        if vector_size > 0:
            return VectorSizeHint(inputCol=col_name, size=vector_size)
    else:
        print(f"No sample found for column '{col_name}'. Skipping VectorSizeHint.")
    
    return None

# Step 3: Create a list of VectorSizeHint stages for each vector column
vector_size_hint_stages = []
for col_name in vector_cols:
    print(f"Getting VectorSizeHint for vector column: '{col_name}'")
    size_hint_stage = get_vector_size_hint_stage(train_data, col_name)
    if size_hint_stage:
        vector_size_hint_stages.append(size_hint_stage)

# Step 4: Combine vector and non-vector columns into a single list for VectorAssembler
final_input_cols = vector_cols + non_vector_cols
print("Final input columns for feature assembly:", final_input_cols)

# Step 5: Assemble final features column using VectorAssembler
final_assembler = VectorAssembler(inputCols=final_input_cols, outputCol="features")

# Step 6: Initialize StandardScaler
standardScaler = StandardScaler(inputCol="features", outputCol="features_scaled")

# Step 7: Create a pipeline with VectorSizeHint stages, VectorAssembler, and StandardScaler
pipeline_stages = vector_size_hint_stages + [final_assembler, standardScaler]
pipeline_final = Pipeline(stages=pipeline_stages)

# Step 8: Fit the pipeline on train_data
model_final = pipeline_final.fit(train_data)
print("Pipeline fitting done.")

# Step 9: Transform train, valid, and test datasets using the fitted pipeline
train_data = model_final.transform(train_data)
valid_data = model_final.transform(valid_data)
test_data = model_final.transform(test_data)
print("Pipeline transformation done.")

Getting VectorSizeHint for vector column: 'gender_onehot'
Column 'gender_onehot' vector size: 4
Getting VectorSizeHint for vector column: 'race_onehot'
Column 'race_onehot' vector size: 7
Final input columns for feature assembly: ['gender_onehot', 'race_onehot', 'age_of_TBI_diagnosis', 'MedicalHistory', 'Z21', 'M19', 'S68', 'Y30', 'B05', 'A23', 'H82', 'V89', 'I31', 'V72', 'R16', 'Q61', 'O12', 'X76', 'Z12', 'S39', 'L65', 'F25', 'G12', 'E02', 'X04', 'B79', 'F32', 'M54', 'B34', 'S60', 'Z19', 'T36', 'E44', 'E83', 'Q65', 'R71', 'D66', 'Z64', 'J60', 'D81', 'F21', 'Q14', 'C22', 'P00', 'B01', 'Z49', 'V24', 'R13', 'I44', 'O68', 'L74', 'D28', 'E56', 'H59', 'I01', 'R06', 'O14', 'K56', 'C78', 'R37', 'A46', 'Y02', 'P72', 'K62', 'S00', 'M92', 'K40', 'M46', 'R80', 'C95', 'K03', 'C24', 'J63', 'X13', 'P13', 'H51', 'I21', 'C77', 'Z08', 'D16', 'O11', 'I06', 'F99', 'J93', 'F01', 'A92', 'P96', 'H34', 'E35', 'Q62', 'W42', 'Z91', 'G96', 'I63', 'B39', 'Q80', 'W58', 'I77', 'R47', 'N99', 'L27', 'J81', 'D21', 'B

Pipeline fitting done.
Pipeline transformation done.


In [6]:
#Step7
train_data = train_data.withColumn('label',train_data.label.cast('double'))
valid_data = valid_data.withColumn('label',valid_data.label.cast('double'))
test_data = test_data.withColumn('label',test_data.label.cast('double'))

In [7]:
from pyspark.sql.functions import col

# Count the number of 0's and 1's in the label column
count_zeros = train_data.filter(col('label') == 0).count()
count_ones = train_data.filter(col('label') == 1).count()

# Separate the majority and minority classes
majority_class_df = train_data.filter(col('label') == 0)
minority_class_df = train_data.filter(col('label') == 1)

# Check if we need to upsample or downsample
if count_zeros > count_ones:
    # Upsample the minority class
    upsample_ratio = count_zeros // count_ones  # Calculate integer part of ratio
    remaining_samples_fraction = (count_zeros % count_ones) / count_ones  # Remaining fraction

    # Duplicate the minority class to match the majority class count
    upsampled_minority_class_df = minority_class_df
    for _ in range(upsample_ratio - 1):  # -1 because we already have one instance
        upsampled_minority_class_df = upsampled_minority_class_df.union(minority_class_df)

    # Add the remaining samples to reach exact count
    upsampled_minority_class_df = upsampled_minority_class_df.union(
        minority_class_df.sample(withReplacement=True, fraction=remaining_samples_fraction)
    )
    
    # Combine with majority class
    train_data_balanced = majority_class_df.union(upsampled_minority_class_df)

else:
    # Downsample the majority class if count_ones > count_zeros
    downsample_fraction = count_ones / count_zeros
    downsampled_majority_class_df = majority_class_df.sample(withReplacement=False, fraction=downsample_fraction)
    
    # Combine with minority class
    train_data_balanced = downsampled_majority_class_df.union(minority_class_df)

# Display the counts after balancing
print("Number of 0's in the balanced DataFrame:", train_data_balanced.filter(col('label') == 0).count())
print("Number of 1's in the balanced DataFrame:", train_data_balanced.filter(col('label') == 1).count())

Number of 0's in the balanced DataFrame: 70345
Number of 1's in the balanced DataFrame: 70431


In [10]:
#Step10
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.mllib.evaluation import BinaryClassificationMetrics, MulticlassMetrics
from pyspark.sql import DataFrame

# Define hyperparameter grid for GBTClassifier
param_grid = {
    "maxDepth": [5, 10],
    "maxIter": [10, 20]
}

# Function to perform hyperparameter tuning and model selection
def tune_gbt_model(train_data_balanced: DataFrame, valid_data: DataFrame, param_grid: dict, label_col: str):
    evaluator = BinaryClassificationEvaluator(labelCol=label_col, rawPredictionCol="rawPrediction", metricName="areaUnderROC")
    
    best_model = None
    best_auc = 0.0
    best_params = {}

    # Loop through all combinations of hyperparameters
    for max_depth in param_grid["maxDepth"]:
        for max_iter in param_grid["maxIter"]:
            # Instantiate a new GBTClassifier for each hyperparameter combination
            gbt = GBTClassifier(labelCol=label_col, featuresCol="features_scaled", maxDepth=max_depth, maxIter=max_iter)

            # Fit the model on the training data
            model = gbt.fit(train_data_balanced)

            # Evaluate the model on the validation data
            valid_predictions = model.transform(valid_data)
            validation_auc = evaluator.evaluate(valid_predictions)

            # Calculate training predictions for accuracy
            train_predictions = model.transform(train_data_balanced)
            train_accuracy = train_predictions.filter(train_predictions.label == train_predictions.prediction).count() / float(train_predictions.count())

            # Print current parameters and validation AUC
            print(f"MaxDepth: {max_depth}, MaxIter: {max_iter}, Training Accuracy: {train_accuracy}, Validation AUC: {validation_auc}")

            # Check if this is the best model
            if validation_auc > best_auc:
                best_auc = validation_auc
                best_model = model
                best_params = {"maxDepth": max_depth, "maxIter": max_iter}

    return best_model, best_params, best_auc

# Call the function to tune the GBT model
best_model, best_params, best_auc = tune_gbt_model(train_data_balanced, valid_data, param_grid, label_col="label")

# Print the best parameters and validation AUC
print(f"Best Parameters: {best_params}")
print(f"Best Validation AUC: {best_auc}")

# Make predictions on the test data using the best model found
test_predictions = best_model.transform(test_data)

# Prepare for evaluation by selecting label and prediction columns
predictionAndTarget = test_predictions.select("label", "prediction")
predictionAndTarget = predictionAndTarget.withColumn('label', predictionAndTarget.label.cast('double'))

# Convert the DataFrame to an RDD and compute the metrics
predictionAndTarget_rdd = predictionAndTarget.rdd.map(tuple)

# Binary and Multiclass metrics
metrics_binary = BinaryClassificationMetrics(predictionAndTarget_rdd)
metrics_multi = MulticlassMetrics(predictionAndTarget_rdd)

# Example: Print AUC and Accuracy
print(f"Test AUC: {metrics_binary.areaUnderROC}")
print(f"Test Accuracy: {metrics_multi.accuracy}")

MaxDepth: 5, MaxIter: 10, Training Accuracy: 0.7910089029211821, Validation AUC: 0.8737301582684305
MaxDepth: 5, MaxIter: 20, Training Accuracy: 0.8050743806354353, Validation AUC: 0.8886143637555909
MaxDepth: 10, MaxIter: 10, Training Accuracy: 0.842819353187132, Validation AUC: 0.8915922836348468
MaxDepth: 10, MaxIter: 20, Training Accuracy: 0.8656242000170663, Validation AUC: 0.9023440067353211
Best Parameters: {'maxDepth': 10, 'maxIter': 20}
Best Validation AUC: 0.9023440067353211
Test AUC: 0.738909417128171
Test Accuracy: 0.8754240162822252


In [11]:
#Step11
###################Using Balanced Samples with Single Validation Set& multiple param ###############################################
confusion_matrix = metrics_multi.confusionMatrix().toArray()
tn, fp, fn, tp = confusion_matrix[0][0], confusion_matrix[0][1], confusion_matrix[1][0], confusion_matrix[1][1]
sensitivity = tp / (fn + tp)
specificity = tn / (tn + fp)

# Compute other metrics
accuracy = metrics_multi.accuracy
f1_1 = metrics_multi.fMeasure(1.0)
f1_0 = metrics_multi.fMeasure(0.0)
precision_1 = metrics_multi.precision(1.0)
precision_0 = metrics_multi.precision(0.0)
recall_1 = metrics_multi.recall(1.0)
recall_0 = metrics_multi.recall(0.0)
auc = metrics_binary.areaUnderROC
# Print the confusion matrix with labels
print("Confusion Matrix:")
print(f"          Predicted: 0     Predicted: 1")
print(f"Actual: 0     TN: {tn}         FP: {fp}")
print(f"Actual: 1     FN: {fn}         TP: {tp}")
print(confusion_matrix)
# Print the metrics
print("\nMetrics:")
print("f1_1: ", f1_1)
print("f1_0: ", f1_0)
print("precision_1: ", precision_1)
print("precision_0: ", precision_0)
print("recall_1: ", recall_1)
print("recall_0: ", recall_0)
print("auc: ", auc)
print("accuracy: ", accuracy)
print('sensitivity: ', sensitivity)
print('specificity: ', specificity)

Confusion Matrix:
          Predicted: 0     Predicted: 1
Actual: 0     TN: 9182.0         FP: 424.0
Actual: 1     FN: 1045.0         TP: 1141.0
[[9182.  424.]
 [1045. 1141.]]

Metrics:
f1_1:  0.6083711010397227
f1_0:  0.9259315282609791
precision_1:  0.729073482428115
precision_0:  0.8978194974088198
recall_1:  0.5219579139981702
recall_0:  0.9558609202581719
auc:  0.738909417128171
accuracy:  0.8754240162822252
sensitivity:  0.5219579139981702
specificity:  0.9558609202581719


In [9]:
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.mllib.evaluation import BinaryClassificationMetrics, MulticlassMetrics

# Initialize the GBTClassifier
gbt = GBTClassifier(labelCol="label", featuresCol="features_scaled")

# Set up the parameter grid for hyperparameter tuning
paramGrid = ParamGridBuilder() \
    .addGrid(gbt.maxDepth, [5, 10]) \
    .addGrid(gbt.maxIter, [10, 20]) \
    .build()

# Use BinaryClassificationEvaluator for model evaluation with AUC metric
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# Set up the CrossValidator
crossval = CrossValidator(estimator=gbt,
                          estimatorParamMaps=paramGrid,
                          evaluator=evaluator,
                          numFolds=3)  # 3-fold cross-validation

# Fit the model using cross-validation
cv_model = crossval.fit(train_data_balanced)

# Evaluate the best model on the validation data
valid_predictions = cv_model.transform(valid_data)
validation_auc = evaluator.evaluate(valid_predictions)
print(f"Best Validation AUC: {validation_auc}")

# Cast the label column in the test data to double (if necessary)
test_data = test_data.withColumn('label', test_data.label.cast('double'))

# Make predictions on the test data using the best model found by cross-validation
test_predictions = cv_model.transform(test_data)

# Prepare for evaluation by selecting label and prediction columns
predictionAndTarget = test_predictions.select("label", "prediction")
predictionAndTarget = predictionAndTarget.withColumn('label', predictionAndTarget.label.cast('double'))

# Convert the DataFrame to an RDD and compute the metrics
predictionAndTarget_rdd = predictionAndTarget.rdd.map(tuple)

# Binary and Multiclass metrics
metrics_binary = BinaryClassificationMetrics(predictionAndTarget_rdd)
metrics_multi = MulticlassMetrics(predictionAndTarget_rdd)

# Example: Print AUC and Accuracy
print(f"Test AUC: {metrics_binary.areaUnderROC}")
print(f"Test Accuracy: {metrics_multi.accuracy}")

Py4JJavaError: An error occurred while calling o440.fit.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 20 in stage 54.0 failed 1 times, most recent failure: Lost task 20.0 in stage 54.0 (TID 899, localhost, executor driver): java.lang.OutOfMemoryError: Java heap space
	at java.nio.HeapByteBuffer.<init>(HeapByteBuffer.java:57)
	at java.nio.ByteBuffer.allocate(ByteBuffer.java:335)
	at org.apache.spark.sql.execution.columnar.BasicColumnBuilder.initialize(ColumnBuilder.scala:66)
	at org.apache.spark.sql.execution.columnar.NativeColumnBuilder.org$apache$spark$sql$execution$columnar$NullableColumnBuilder$$super$initialize(ColumnBuilder.scala:97)
	at org.apache.spark.sql.execution.columnar.NullableColumnBuilder$class.initialize(NullableColumnBuilder.scala:51)
	at org.apache.spark.sql.execution.columnar.NativeColumnBuilder.org$apache$spark$sql$execution$columnar$compression$CompressibleColumnBuilder$$super$initialize(ColumnBuilder.scala:97)
	at org.apache.spark.sql.

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.7-src.zip/py4j/java_gateway.py", line 1159, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.7-src.zip/py4j/java_gateway.py", line 985, in send_command
    response = connection.send_command(command)
  File "/usr/local/spark/python/lib/py4j-0.10.7-src.zip/py4j/java_gateway.py", line 1164, in send_command
    "Error while receiving", e, proto.ERROR_ON_RECEIVE)
py4j.protocol.Py4JNetworkError: Error while receiving
ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.7-src.zip/py4j/java_gateway.py", line 1159, in send_command
    raise Py4JNetworkError("Answ

In [ ]:
#Step11
###################Using Balanced Samples with Single Validation Set& multiple param ###############################################
confusion_matrix = metrics_multi.confusionMatrix().toArray()
tn, fp, fn, tp = confusion_matrix[0][0], confusion_matrix[0][1], confusion_matrix[1][0], confusion_matrix[1][1]
sensitivity = tp / (fn + tp)
specificity = tn / (tn + fp)

# Compute other metrics
accuracy = metrics_multi.accuracy
f1_1 = metrics_multi.fMeasure(1.0)
f1_0 = metrics_multi.fMeasure(0.0)
precision_1 = metrics_multi.precision(1.0)
precision_0 = metrics_multi.precision(0.0)
recall_1 = metrics_multi.recall(1.0)
recall_0 = metrics_multi.recall(0.0)
auc = metrics_binary.areaUnderROC
# Print the confusion matrix with labels
print("Confusion Matrix:")
print(f"          Predicted: 0     Predicted: 1")
print(f"Actual: 0     TN: {tn}         FP: {fp}")
print(f"Actual: 1     FN: {fn}         TP: {tp}")
print(confusion_matrix)
# Print the metrics
print("\nMetrics:")
print("f1_1: ", f1_1)
print("f1_0: ", f1_0)
print("precision_1: ", precision_1)
print("precision_0: ", precision_0)
print("recall_1: ", recall_1)
print("recall_0: ", recall_0)
print("auc: ", auc)
print("accuracy: ", accuracy)
print('sensitivity: ', sensitivity)
print('specificity: ', specificity)

In [ ]:
#############Support Vector Machine (SVM) classifier with an RBF kernel using PySpark ML ##########################################
################# Used LinearSVC class from PySpark since PySpark ML currently does not support RBF kernel SVM directly ############

from pyspark.ml.classification import LinearSVC
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql import DataFrame

# Define hyperparameter grid for SVM (linear approximation as PySpark does not support RBF kernel directly)
param_grid = {
    "maxIter": [10, 20],
    "regParam": [0.1, 0.01]  # Regularization parameter
}

# Function to perform hyperparameter tuning and model selection
def tune_svm_model(train_data: DataFrame, valid_data: DataFrame, param_grid: dict, label_col: str):
    evaluator = BinaryClassificationEvaluator(labelCol=label_col, rawPredictionCol="rawPrediction", metricName="areaUnderROC")
    
    best_model = None
    best_auc = 0.0
    best_params = {}

    # Loop through all combinations of hyperparameters
    for max_iter in param_grid["maxIter"]:
        for reg_param in param_grid["regParam"]:
            # Instantiate a new LinearSVC for each hyperparameter combination
            svm = LinearSVC(labelCol=label_col, featuresCol="features_scaled", maxIter=max_iter, regParam=reg_param)

            # Fit the model on the training data
            model = svm.fit(train_data)

            # Evaluate the model on the validation data
            valid_predictions = model.transform(valid_data)
            validation_auc = evaluator.evaluate(valid_predictions)

            # Calculate training predictions for accuracy
            train_predictions = model.transform(train_data)
            train_accuracy = train_predictions.filter(train_predictions.label == train_predictions.prediction).count() / float(train_predictions.count())

            # Print current parameters and validation AUC
            print(f"MaxIter: {max_iter}, RegParam: {reg_param}, Training Accuracy: {train_accuracy}, Validation AUC: {validation_auc}")

            # Check if this is the best model
            if validation_auc > best_auc:
                best_auc = validation_auc
                best_model = model
                best_params = {"maxIter": max_iter, "regParam": reg_param}

    return best_model, best_params, best_auc

# Call the function to tune the SVM model
best_model, best_params, best_auc = tune_svm_model(train_data_balanced, valid_data, param_grid, label_col="label")

# Print the best parameters and validation AUC
print(f"Best Parameters: {best_params}")
print(f"Best Validation AUC: {best_auc}")

# Make predictions on the test data using the best model found
test_predictions = best_model.transform(test_data)

# Prepare for evaluation by selecting label and prediction columns
predictionAndTarget = test_predictions.select("label", "prediction")
predictionAndTarget = predictionAndTarget.withColumn('label', predictionAndTarget.label.cast('double'))

# Convert the DataFrame to an RDD and compute the metrics
predictionAndTarget_rdd = predictionAndTarget.rdd.map(tuple)

# Binary metrics
from pyspark.mllib.evaluation import BinaryClassificationMetrics, MulticlassMetrics
metrics_binary = BinaryClassificationMetrics(predictionAndTarget_rdd)
metrics_multi = MulticlassMetrics(predictionAndTarget_rdd)

# Example: Print AUC and Accuracy
print(f"Test AUC: {metrics_binary.areaUnderROC}")
print(f"Test Accuracy: {metrics_multi.accuracy}")


In [ ]:
#Step11
###################Using Balanced Samples with Single Validation Set& multiple param ###############################################
confusion_matrix = metrics_multi.confusionMatrix().toArray()
tn, fp, fn, tp = confusion_matrix[0][0], confusion_matrix[0][1], confusion_matrix[1][0], confusion_matrix[1][1]
sensitivity = tp / (fn + tp)
specificity = tn / (tn + fp)

# Compute other metrics
accuracy = metrics_multi.accuracy
f1_1 = metrics_multi.fMeasure(1.0)
f1_0 = metrics_multi.fMeasure(0.0)
precision_1 = metrics_multi.precision(1.0)
precision_0 = metrics_multi.precision(0.0)
recall_1 = metrics_multi.recall(1.0)
recall_0 = metrics_multi.recall(0.0)
auc = metrics_binary.areaUnderROC
# Print the confusion matrix with labels
print("Confusion Matrix:")
print(f"          Predicted: 0     Predicted: 1")
print(f"Actual: 0     TN: {tn}         FP: {fp}")
print(f"Actual: 1     FN: {fn}         TP: {tp}")
print(confusion_matrix)
# Print the metrics
print("\nMetrics:")
print("f1_1: ", f1_1)
print("f1_0: ", f1_0)
print("precision_1: ", precision_1)
print("precision_0: ", precision_0)
print("recall_1: ", recall_1)
print("recall_0: ", recall_0)
print("auc: ", auc)
print("accuracy: ", accuracy)
print('sensitivity: ', sensitivity)
print('specificity: ', specificity)

In [8]:
#######################################RandomForest with class balanced #####################################################
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.mllib.evaluation import BinaryClassificationMetrics, MulticlassMetrics
from pyspark.sql import DataFrame

# Define hyperparameter grid for RandomForestClassifier
param_grid = {
    "maxDepth": [5, 10],
    "numTrees": [10, 20]  # Changed from maxIter to numTrees for RandomForest
}

# Function to perform hyperparameter tuning and model selection
def tune_random_forest_model(train_data_balanced: DataFrame, valid_data: DataFrame, param_grid: dict, label_col: str):
    evaluator = BinaryClassificationEvaluator(labelCol=label_col, rawPredictionCol="rawPrediction", metricName="areaUnderROC")
    
    best_model = None
    best_auc = 0.0
    best_params = {}

    # Loop through all combinations of hyperparameters
    for max_depth in param_grid["maxDepth"]:
        for num_trees in param_grid["numTrees"]:
            # Instantiate a new RandomForestClassifier for each hyperparameter combination
            rf = RandomForestClassifier(labelCol=label_col, featuresCol="features_scaled", maxDepth=max_depth, numTrees=num_trees)

            # Fit the model on the training data
            model = rf.fit(train_data_balanced)

            # Evaluate the model on the validation data
            valid_predictions = model.transform(valid_data)
            validation_auc = evaluator.evaluate(valid_predictions)

            # Calculate training predictions for accuracy
            train_predictions = model.transform(train_data_balanced)
            train_accuracy = train_predictions.filter(train_predictions.label == train_predictions.prediction).count() / float(train_predictions.count())

            # Print current parameters and validation AUC
            print(f"MaxDepth: {max_depth}, NumTrees: {num_trees}, Training Accuracy: {train_accuracy}, Validation AUC: {validation_auc}")

            # Check if this is the best model
            if validation_auc > best_auc:
                best_auc = validation_auc
                best_model = model
                best_params = {"maxDepth": max_depth, "numTrees": num_trees}

    return best_model, best_params, best_auc

# Call the function to tune the Random Forest model
best_model, best_params, best_auc = tune_random_forest_model(train_data_balanced, valid_data, param_grid, label_col="label")

# Print the best parameters and validation AUC
print(f"Best Parameters: {best_params}")
print(f"Best Validation AUC: {best_auc}")

# Make predictions on the test data using the best model found
test_predictions = best_model.transform(test_data)

# Prepare for evaluation by selecting label and prediction columns
predictionAndTarget = test_predictions.select("label", "prediction")
predictionAndTarget = predictionAndTarget.withColumn('label', predictionAndTarget.label.cast('double'))

# Convert the DataFrame to an RDD and compute the metrics
predictionAndTarget_rdd = predictionAndTarget.rdd.map(tuple)

# Binary and Multiclass metrics
metrics_binary = BinaryClassificationMetrics(predictionAndTarget_rdd)
metrics_multi = MulticlassMetrics(predictionAndTarget_rdd)

# Example: Print AUC and Accuracy
print(f"Test AUC: {metrics_binary.areaUnderROC}")
print(f"Test Accuracy: {metrics_multi.accuracy}")

MaxDepth: 5, NumTrees: 10, Training Accuracy: 0.7585525941921919, Validation AUC: 0.8234614019675716
MaxDepth: 5, NumTrees: 20, Training Accuracy: 0.770941069500483, Validation AUC: 0.8338336149413772
MaxDepth: 10, NumTrees: 10, Training Accuracy: 0.7793444905381599, Validation AUC: 0.8423565565084704
MaxDepth: 10, NumTrees: 20, Training Accuracy: 0.78862874353583, Validation AUC: 0.8523333352725917
Best Parameters: {'maxDepth': 10, 'numTrees': 20}
Best Validation AUC: 0.8523333352725917
Test AUC: 0.6706972931615941
Test Accuracy: 0.8139416553595658


In [9]:
#Step11
###################Using Balanced Samples with Single Validation Set& multiple param ###############################################
confusion_matrix = metrics_multi.confusionMatrix().toArray()
tn, fp, fn, tp = confusion_matrix[0][0], confusion_matrix[0][1], confusion_matrix[1][0], confusion_matrix[1][1]
sensitivity = tp / (fn + tp)
specificity = tn / (tn + fp)

# Compute other metrics
accuracy = metrics_multi.accuracy
f1_1 = metrics_multi.fMeasure(1.0)
f1_0 = metrics_multi.fMeasure(0.0)
precision_1 = metrics_multi.precision(1.0)
precision_0 = metrics_multi.precision(0.0)
recall_1 = metrics_multi.recall(1.0)
recall_0 = metrics_multi.recall(0.0)
auc = metrics_binary.areaUnderROC
# Print the confusion matrix with labels
print("Confusion Matrix:")
print(f"          Predicted: 0     Predicted: 1")
print(f"Actual: 0     TN: {tn}         FP: {fp}")
print(f"Actual: 1     FN: {fn}         TP: {tp}")
print(confusion_matrix)
# Print the metrics
print("\nMetrics:")
print("f1_1: ", f1_1)
print("f1_0: ", f1_0)
print("precision_1: ", precision_1)
print("precision_0: ", precision_0)
print("recall_1: ", recall_1)
print("recall_0: ", recall_0)
print("auc: ", auc)
print("accuracy: ", accuracy)
print('sensitivity: ', sensitivity)
print('specificity: ', specificity)

Confusion Matrix:
          Predicted: 0     Predicted: 1
Actual: 0     TN: 8473.0         FP: 440.0
Actual: 1     FN: 1754.0         TP: 1125.0
[[8473.  440.]
 [1754. 1125.]]

Metrics:
f1_1:  0.5063006300630064
f1_0:  0.8853709508881923
precision_1:  0.7188498402555911
precision_0:  0.8284932042632248
recall_1:  0.39076068079194165
recall_0:  0.9506339055312465
auc:  0.6706972931615941
accuracy:  0.8139416553595658
sensitivity:  0.39076068079194165
specificity:  0.9506339055312465


In [ ]:
#######################################DecisionTree with GridSearchCV #####################################################
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import DataFrame

# Function to perform hyperparameter tuning and model selection
def tune_dt_model(train_data_balanced: DataFrame, valid_data: DataFrame, label_col: str):
    # Initialize the DecisionTreeClassifier
    dt = DecisionTreeClassifier(labelCol=label_col, featuresCol="features_scaled")

    # Define the evaluator with areaUnderROC as the metric
    evaluator = BinaryClassificationEvaluator(labelCol=label_col, rawPredictionCol="rawPrediction", metricName="areaUnderROC")

    # Define the parameter grid for hyperparameter tuning
    param_grid = (ParamGridBuilder()
                  .addGrid(dt.maxDepth, [5, 10])  # Maximum depth of the tree
                  .addGrid(dt.minInfoGain, [0.0, 0.1])  # Minimum info gain
                  .build())

    # Set up CrossValidator for hyperparameter tuning
    crossval = CrossValidator(estimator=dt,
                              estimatorParamMaps=param_grid,
                              evaluator=evaluator,
                              numFolds=3)  # Adjust numFolds as needed

    # Fit CrossValidator on training data
    cv_model = crossval.fit(train_data_balanced)

    # Get the best model from CrossValidator
    best_model = cv_model.bestModel

    # Extract the best hyperparameters
    best_params = {
        "maxDepth": best_model._java_obj.getMaxDepth(),
        "minInfoGain": best_model._java_obj.getMinInfoGain()
    }

    # Evaluate best model on validation data
    best_auc = evaluator.evaluate(best_model.transform(valid_data))

    # Print best parameters and validation AUC
    print(f"Best Parameters: {best_params}")
    print(f"Best Validation AUC: {best_auc}")

    return best_model, best_params, best_auc

# Example call to the function
best_model, best_params, best_auc = tune_dt_model(train_data_balanced, valid_data, label_col="label")

# Make predictions on the test data using the best model found
test_predictions = best_model.transform(test_data)

# Prepare for evaluation by selecting label and prediction columns
predictionAndTarget = test_predictions.select("label", "prediction")
predictionAndTarget = predictionAndTarget.withColumn("label", predictionAndTarget.label.cast("double"))

# Convert the DataFrame to an RDD and compute the metrics
predictionAndTarget_rdd = predictionAndTarget.rdd.map(tuple)

# Binary and Multiclass metrics
from pyspark.mllib.evaluation import BinaryClassificationMetrics, MulticlassMetrics
metrics_binary = BinaryClassificationMetrics(predictionAndTarget_rdd)
metrics_multi = MulticlassMetrics(predictionAndTarget_rdd)

# Example: Print AUC and Accuracy
print(f"Test AUC: {metrics_binary.areaUnderROC}")
print(f"Test Accuracy: {metrics_multi.accuracy}")

In [ ]:
#Step11
###################Using Balanced Samples with Single Validation Set& multiple param ###############################################
confusion_matrix = metrics_multi.confusionMatrix().toArray()
tn, fp, fn, tp = confusion_matrix[0][0], confusion_matrix[0][1], confusion_matrix[1][0], confusion_matrix[1][1]
sensitivity = tp / (fn + tp)
specificity = tn / (tn + fp)

# Compute other metrics
accuracy = metrics_multi.accuracy
f1_1 = metrics_multi.fMeasure(1.0)
f1_0 = metrics_multi.fMeasure(0.0)
precision_1 = metrics_multi.precision(1.0)
precision_0 = metrics_multi.precision(0.0)
recall_1 = metrics_multi.recall(1.0)
recall_0 = metrics_multi.recall(0.0)
auc = metrics_binary.areaUnderROC
# Print the confusion matrix with labels
print("Confusion Matrix:")
print(f"          Predicted: 0     Predicted: 1")
print(f"Actual: 0     TN: {tn}         FP: {fp}")
print(f"Actual: 1     FN: {fn}         TP: {tp}")
print(confusion_matrix)
# Print the metrics
print("\nMetrics:")
print("f1_1: ", f1_1)
print("f1_0: ", f1_0)
print("precision_1: ", precision_1)
print("precision_0: ", precision_0)
print("recall_1: ", recall_1)
print("recall_0: ", recall_0)
print("auc: ", auc)
print("accuracy: ", accuracy)
print('sensitivity: ', sensitivity)
print('specificity: ', specificity)

In [ ]:
#######################################XGBT with GridSearchCV #####################################################
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import DataFrame
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import GBTClassifier
from sparkxgb import XGBoostClassifier  # Make sure you have sparkxgb installed

# Function to perform hyperparameter tuning and model selection for XGBoost
def tune_xgb_model(train_data_balanced: DataFrame, valid_data: DataFrame, label_col: str):
    # Initialize the XGBoost classifier
    xgb = XGBoostClassifier(
        labelCol=label_col,
        featuresCol="features_scaled",
        objective='binary:logistic',
        eval_metric='auc'
    )

    # Define the evaluator with areaUnderROC as the metric
    evaluator = BinaryClassificationEvaluator(labelCol=label_col, metricName="areaUnderROC")

    # Define the parameter grid for hyperparameter tuning
    param_grid = (ParamGridBuilder()
                  .addGrid(xgb.maxDepth, [5, 10])  # Maximum depth of trees
                  .addGrid(xgb.numRounds, [100, 200])  # Number of boosting rounds
                  .build())

    # Set up CrossValidator for hyperparameter tuning
    crossval = CrossValidator(estimator=xgb,
                              estimatorParamMaps=param_grid,
                              evaluator=evaluator,
                              numFolds=3)  # Adjust numFolds as needed

    # Fit CrossValidator on training data
    cv_model = crossval.fit(train_data_balanced)

    # Get the best model from CrossValidator
    best_model = cv_model.bestModel

    # Extract the best hyperparameters
    best_params = {
        "maxDepth": best_model.getMaxDepth(),
        "numRounds": best_model.getNumRounds()
    }

    # Evaluate best model on validation data
    best_auc = evaluator.evaluate(best_model.transform(valid_data))

    # Print best parameters and validation AUC
    print(f"Best Parameters: {best_params}")
    print(f"Best Validation AUC: {best_auc}")

    return best_model, best_params, best_auc

# Example call to the function
best_model, best_params, best_auc = tune_xgb_model(train_data_balanced, valid_data, label_col="label")

# Make predictions on the test data using the best model found
test_predictions = best_model.transform(test_data)

# Prepare for evaluation by selecting label and prediction columns
predictionAndTarget = test_predictions.select("label", "prediction")
predictionAndTarget = predictionAndTarget.withColumn("label", predictionAndTarget.label.cast("double"))

# Convert the DataFrame to an RDD and compute the metrics
predictionAndTarget_rdd = predictionAndTarget.rdd.map(tuple)

# Binary and Multiclass metrics
from pyspark.mllib.evaluation import BinaryClassificationMetrics, MulticlassMetrics
metrics_binary = BinaryClassificationMetrics(predictionAndTarget_rdd)
metrics_multi = MulticlassMetrics(predictionAndTarget_rdd)

# Example: Print AUC and Accuracy
print(f"Test AUC: {metrics_binary.areaUnderROC}")
print(f"Test Accuracy: {metrics_multi.accuracy}")

In [ ]:
#Step11
###################Using Balanced Samples with Single Validation Set& multiple param ###############################################
confusion_matrix = metrics_multi.confusionMatrix().toArray()
tn, fp, fn, tp = confusion_matrix[0][0], confusion_matrix[0][1], confusion_matrix[1][0], confusion_matrix[1][1]
sensitivity = tp / (fn + tp)
specificity = tn / (tn + fp)

# Compute other metrics
accuracy = metrics_multi.accuracy
f1_1 = metrics_multi.fMeasure(1.0)
f1_0 = metrics_multi.fMeasure(0.0)
precision_1 = metrics_multi.precision(1.0)
precision_0 = metrics_multi.precision(0.0)
recall_1 = metrics_multi.recall(1.0)
recall_0 = metrics_multi.recall(0.0)
auc = metrics_binary.areaUnderROC
# Print the confusion matrix with labels
print("Confusion Matrix:")
print(f"          Predicted: 0     Predicted: 1")
print(f"Actual: 0     TN: {tn}         FP: {fp}")
print(f"Actual: 1     FN: {fn}         TP: {tp}")
print(confusion_matrix)
# Print the metrics
print("\nMetrics:")
print("f1_1: ", f1_1)
print("f1_0: ", f1_0)
print("precision_1: ", precision_1)
print("precision_0: ", precision_0)
print("recall_1: ", recall_1)
print("recall_0: ", recall_0)
print("auc: ", auc)
print("accuracy: ", accuracy)
print('sensitivity: ', sensitivity)
print('specificity: ', specificity)